In [3]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()

True

In [ ]:
# https://z.ai/blog/glm-5#:~:text=Humanity's%20Last%20Exam%20(HLE)%20&,task%20CPU%20and%20memory%20constraints.
# https://github.com/MiniMax-AI/MiniMax-M2
# https://huggingface.co/Qwen/Qwen3-Coder-30B-A3B-Instruct#:~:text=We%20recommend%20using%20an%20output%20length%20of,queries%2C%20which%20is%20adequate%20for%20instruct%20models.

In [5]:
from mlx_lm import load, generate

# model_path = "../models/Qwen2.5-3B-Instruct-4bit"
# model_path = "../models/gemma-4-26b-a4b-8bit"
model_path = "../models/gemma-4-transformers-gemma-4-26b-a4b-it-v1"
model, tokenizer = load(model_path)

# Standard Chat Template
# messages = [{"role": "user", "content": "Explain how to use function calling in Qwen."}]
messages = [{"role": "user", "content": "Explain how to use function calling in Gemma."}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Generate response
response = generate(model, tokenizer, prompt=prompt, verbose=True, max_tokens=8012)

You are using a model of type `gemma4` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


<|channel>thought

*   *Subject:* Function calling in Gemma.
*   *Goal:* Explain *how* to use it.
*   *Context:* Gemma is an open-weights model family from Google. Unlike proprietary models (like GPT-4 or Gemini) that have built-in, API-level function calling support, Gemma is a base/instruction model. To use function calling with Gemma, you have to implement the "loop" yourself using prompting and parsing.

    *   Gemma doesn't have a native `tools` parameter in a standard API like OpenAI's `tools`.
    *   Instead, it relies on **Prompt Engineering** (System Prompts) and **Structured Output** (JSON).
    *   The workflow is:
        1.  Define functions (JSON schema).
        2.  Provide these definitions in the system prompt.
        3.  Model generates a "call" (a JSON string).
        4.  The *application* (the code running the model) executes the function.
        5.  The *application* feeds the result back to the model.
        6.  The model generates a final natural language r

In [ ]:
# speculative decoding

In [ ]:
import mlx.core as mx
from mlx_lm import load, generate
from mlx_lm.utils import load_config
from mlx_lm.sample_utils import make_sampler

# 1. Configuration - Set your paths here
MODEL_PATH = "../Qwen2.5-3B-Instruct-4bit"
# DRAFT_MODEL_PATH = "../Qwen2.5-0.5B-Instruct-4bit" # Future upgrade

# 2. Load Model & Tokenizer
# On M5 Max, this utilizes the Fusion Architecture's zero-copy loading
model, tokenizer = load(MODEL_PATH)

# 3. Define an Agentic Prompt with Function Calling
def test_agent_logic(user_query):
    messages = [
        {"role": "system", "content": "You are ContainerClaw Core. You have access to local shell tools."},
        {"role": "user", "content": user_query}
    ]
    
    prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )

    print(f"--- Generating Response (Model: {MODEL_PATH}) ---")
    
    # We use stream_generate for real-time feedback (crucial for agents)
    # verbose=True shows the TPS and memory usage stats
    # Create a dedicated sampler object
    # sampler = make_sampler(temp=0.7, top_p=0.8, top_k=20,)
    sampler = make_sampler(temp=0.7, top_p=0.95, )

    response = generate(
        model, 
        tokenizer, 
        prompt=prompt, 
        max_tokens=512,
        sampler=sampler, # Pass the sampler here
        verbose=True 
    )
    return response

# 4. Run Test
query = "Create a python script to check the thermal throttling status of this M5 Max."
test_agent_logic(query)

--- Generating Response (Model: ../Qwen2.5-3B-Instruct-4bit) ---
To check the thermal throttling status of an Apple M5 Max GPU, we need to look at the performance counter data, as the M5 Max is an integrated GPU with performance counters exposed through various interfaces. For this, we will need to use a Linux kernel module or a user-space tool that can access these performance counters.

Given the complexity and the specific nature of the M5 Max GPU, we can use a user-space tool like `perf` (Performance counter tool) to monitor the thermal throttling status. Below is a Python script that uses `perf` to check the thermal throttling status:

1. First, you need to install `perf` if you haven't already. `perf` is usually included with the Linux kernel, but ensure it's installed on your system.
2. Use the following Python script to monitor the performance counters and detect thermal throttling.

```python
import subprocess
import time

# Function to get performance counter data
def get_per

'To check the thermal throttling status of an Apple M5 Max GPU, we need to look at the performance counter data, as the M5 Max is an integrated GPU with performance counters exposed through various interfaces. For this, we will need to use a Linux kernel module or a user-space tool that can access these performance counters.\n\nGiven the complexity and the specific nature of the M5 Max GPU, we can use a user-space tool like `perf` (Performance counter tool) to monitor the thermal throttling status. Below is a Python script that uses `perf` to check the thermal throttling status:\n\n1. First, you need to install `perf` if you haven\'t already. `perf` is usually included with the Linux kernel, but ensure it\'s installed on your system.\n2. Use the following Python script to monitor the performance counters and detect thermal throttling.\n\n```python\nimport subprocess\nimport time\n\n# Function to get performance counter data\ndef get_performance_counter():\n    perf_output = subprocess.

In [17]:
import time

queries = [
    "Analyze the current directory structure.",
    "Initialize a git repository in /tmp/claw_test.",
    "Write a Dockerfile for a Rust-based microservice.",
    "Explain the difference between PagedAttention and standard KV caching."
]

for i, q in enumerate(queries):
    start = time.perf_counter()
    _ = generate(model, tokenizer, prompt=q, max_tokens=100)
    end = time.perf_counter()
    print(f"Query {i+1} completed in {end - start:.2f}s")

Query 1 completed in 0.57s
Query 2 completed in 0.53s
Query 3 completed in 0.53s
Query 4 completed in 0.53s


In [32]:
import requests

resp = requests.post(
    "http://127.0.0.1:7860/generate",
    json={"prompt": "Explain function calling in Qwen 2.5."}
)
print(resp.json()["output"])

How does it differ from other models? Qwen 2.5 is a large language model developed by Qwen, and it is designed to be more efficient and faster than previous versions. One of the key differences between Qwen 2.5 and other models is its function calling capability. Function calling allows the model to execute external code and perform tasks that are not directly related to language generation. This is achieved through a combination of pre-processing and post-processing steps, as well as a specialized function calling mechanism.

When a function is called, Qwen 2.5 first preprocesses the input to ensure that it is in a format that can be executed by the external code. This may involve tokenization, special character handling, and other preprocessing steps. Once the input is preprocessed, Qwen 2.5 executes the external code and captures the output. The output is then post-processed to ensure that it is in a format that can be used by the language model.

The function callingcalling mechani

In [42]:
import asyncio
from openai import AsyncOpenAI

# Connect to your running MLX server
client = AsyncOpenAI(
    base_url="http://127.0.0.1:8080/v1",  # include /v1
    api_key="mlx-is-cool"
)

async def run_prompts(prompts):
    results = []
    for prompt in prompts:
        print(f"🚀 Sending: {prompt[:30]}...")
        response = await client.chat.completions.create(
            model="Qwen2.5-3B-Instruct-4bit",
            messages=[{"role": "user", "content": prompt}]
        )
        assistant_msg = response.choices[0].message.content
        results.append(assistant_msg)
        print(f"✅ Response: {assistant_msg[:50]}...\n")
    return results

# Test prompts
test_prompts = ["Hello, MLX!", "Tell me a joke.", "What is 2+2?"]

# If running in Jupyter, use this to schedule the coroutine
results = await run_prompts(test_prompts)
print("All results:", results)

🚀 Sending: Hello, MLX!...


INFO:httpx:HTTP Request: POST http://127.0.0.1:8080/v1/chat/completions "HTTP/1.0 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:8080/v1/chat/completions "HTTP/1.0 200 OK"


✅ Response: Hello! I'm Qwen, created by Alibaba Cloud. How can...

🚀 Sending: Tell me a joke....
✅ Response: Sure! Here's one for you:

Why don't scientists tr...

🚀 Sending: What is 2+2?...


INFO:httpx:HTTP Request: POST http://127.0.0.1:8080/v1/chat/completions "HTTP/1.0 200 OK"


✅ Response: 2 + 2 equals 4....

All results: ["Hello! I'm Qwen, created by Alibaba Cloud. How can I assist you today?", "Sure! Here's one for you:\n\nWhy don't scientists trust atoms?\n\nBecause they make up everything!", '2 + 2 equals 4.']


In [48]:
import asyncio
from openai import AsyncOpenAI

client = AsyncOpenAI(base_url="http://127.0.0.1:8080/v1", api_key="mlx-is-cool")

async def get_response(prompt: str) -> str:
    response = await client.chat.completions.create(
        model="Qwen2.5-3B-Instruct-4bit",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

async def run_parallel(prompts):
    tasks = [get_response(p) for p in prompts]
    return await asyncio.gather(*tasks)

# Example prompts
prompts = ["Hello! Tell me all about yourself.", "Tell me a joke. Make it a long story joke.", "Describe irrational numbers in depth."]

# Run in Jupyter or async environment
results = await run_parallel(prompts)
print(results)

INFO:httpx:HTTP Request: POST http://127.0.0.1:8080/v1/chat/completions "HTTP/1.0 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:8080/v1/chat/completions "HTTP/1.0 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:8080/v1/chat/completions "HTTP/1.0 200 OK"


["Hello! I'm Qwen, a large language model created by Alibaba Cloud. I'm designed to understand and generate human-like text based on the data I was trained on. I don't have a physical body or a specific physical form, but I exist as a software program. I'm here to assist you with various tasks and questions to the best of my ability based on the information and knowledge I've been trained on. How can I assist you today?", 'Certainly! Here’s a long story joke for you:\n\nOnce upon a time, in a faraway land, there was a king who was known for his love of puzzles and riddles. He had a special advisor who was also very good at solving these puzzles. The king decided to challenge his advisor with a particularly tricky riddle. He said, "If I were to give you a kingdom, how many would you need to divide it into to make it yours?"\n\nThe advisor thought for a moment and replied, "Two, sire. One for you, and one for me."\n\nThe king was pleased with the advisor\'s answer, but he wanted to test 

In [49]:
import time

start = time.time()
results = await run_parallel(prompts)
end = time.time()
print(f"Total time: {end - start:.2f} seconds")

INFO:httpx:HTTP Request: POST http://127.0.0.1:8080/v1/chat/completions "HTTP/1.0 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:8080/v1/chat/completions "HTTP/1.0 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:8080/v1/chat/completions "HTTP/1.0 200 OK"


Total time: 2.94 seconds


In [50]:
async def run_sequential(prompts):
    results = []
    for p in prompts:
        results.append(await get_response(p))
    return results

In [51]:
import time

start = time.time()
results = await run_parallel(prompts)
end = time.time()
print(f"Total time: {end - start:.2f} seconds")

INFO:httpx:HTTP Request: POST http://127.0.0.1:8080/v1/chat/completions "HTTP/1.0 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:8080/v1/chat/completions "HTTP/1.0 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:8080/v1/chat/completions "HTTP/1.0 200 OK"


Total time: 2.93 seconds


In [ ]:
# python -m mlx_lm server --model ./Qwen2.5-3B-Instruct-4bit --port 8080 --host 127.0.0.1